### Dynamisches Programmieren - Tabulation

Die Anzahl der Rekursionen ist begrenzt, kann aber erhöht werden.

In [ ]:
import sys
sys.getrecursionlimit()

In [ ]:
sys.setrecursionlimit(10**6)

Meist etwas schneller ist die nicht-rekursive Variante: Tabulation. Die Ergebnisse, die die Rekursion (top-down) liefert, werden in einer Tabelle (bottum-up) aufgebaut.

In [ ]:
def dp(i):
    '''
    returns: die minimalen Kosten für a[i:]. i=0..n-1
    dp-guess: wie groß war der erste Sprung?
    '''
    if i in memo: return memo[i]
        
    if i == n-1: return 0
    if i == n-2: return abs(a[n-1]-a[n-2])
 
    res = min(abs(a[i]-a[i+1]) + dp(i+1), abs(a[i]-a[i+2]) + dp(i+2))
       
    memo[i] = res
    return res

a = [30, 10, 60, 10, 60, 50]
n = len(a)
memo = {}
print(dp(0))

In [ ]:
dp = [0]*len(a)
dp[n-1] = 0
dp[n-2] = abs(a[n-1]-a[n-2])
for i in range(n-3,-1,-1):
    res = min(abs(a[i]-a[i+1]) + dp[i+1], abs(a[i]-a[i+2]) + dp[i+2])
    dp[i] = res
dp[0]
    

#### Cooldown

In [20]:
def dp(i, k):
    if (i,k) in memo: return memo[(i,k)]
        
    if i == n: return 0

    if k == 0:
        res = max(dp(i+1,0),dp(i+1,2) - a[i])
    elif k == 1:
        res = dp(i+1,0)
    elif k == 2:
        res = max(dp(i+1,2),dp(i+1,1) + a[i])
        
    memo[(i,k)] = res
    return res

memo = dict()
a = [1,2,3,0,2]
n = len(a)
dp(0,0)

3

In [21]:
dp = [[0]*3 for _ in range(n+1)]
dp[n][0] = 0
dp[n][1] = 0
dp[n][2] = 0

for i in range(n-1,-1,-1):
    for k in range(3):
        if k == 0:
            res = max(dp[i+1][0],dp[i+1][2] - a[i])
        elif k == 1:
            res = dp[i+1][0]
        elif k == 2:
            res = max(dp[i+1][2],dp[i+1][1] + a[i])
        dp[i][k] = res

dp[0][0]

3

### Dynamisches Programmieren - Parent Pointer

Parent Pointer nutzen wir, wenn uns außer dem Ergebnis auch der Weg zum Ergebnis interessiert.

#### Cooldown

In [22]:
def dp(i, k):
    '''
    returns: maximaler Gewinn für a[i:], wenn zu Beginn im Zustand k, i = 0...n
        k = 0: keine Aktie vorhanden, nicht in cooldown
        k = 1: keine Aktie vorhanden, im cooldown
        k = 2: Aktie vorhanden

    dp-guess: Was machen wir am Tag i, wenn wir uns im Zustand k befinden?
    '''
    if (i,k) in memo: return memo[(i,k)]
        
    if i == n: return 0

    if k == 0:
        res = max(dp(i+1,0),dp(i+1,2) - a[i])
    elif k == 1:
        res = dp(i+1,0)
    elif k == 2:
        res = max(dp(i+1,2),dp(i+1,1) + a[i])
        
    memo[(i,k)] = res
    return res


memo = dict()
a = [48, 12, 60, 93, 97, 42, 25, 64, 17, 56, 85, 93, 9, 48, 52, 42, 58, 85, 81, 84, 69, 36, 1, 54, 23, 15, 72, 15, 11, 94]
n = len(a)
dp(0,0)

428

Dasselbe Programm mit parent-pointern:

In [23]:
def dp(i, k):
    
    if (i,k) in memo: return memo[(i,k)]
        
    if i == n: 
        prev[(i,k)] = None
        return 0
        
    if k == 0:
        res = dp(i+1,0)
        prev[(i,k)] = (i+1,0)
        if dp(i+1,2) - a[i] > res:
            res = dp(i+1,2) - a[i]
            prev[(i,k)] = (i+1,2)
    elif k == 1:
        res = dp(i+1,0)
        prev[(i,k)] = (i+1,0)
    elif k == 2:
        res = dp(i+1,2)
        prev[(i,k)] = (i+1,2)
        if dp(i+1,1) + a[i] > res:
            res =  dp(i+1,1) + a[i] 
            prev[(i,k)] = (i+1,1)
       
    memo[(i,k)] = res
    return res

prev = dict()
memo = dict()

a = [48, 12, 60, 93, 97, 42, 25, 64, 17, 56, 85, 93, 9, 48, 52, 42, 58, 85, 81, 84, 69, 36, 1, 54, 23, 15, 72, 15, 11, 94]
n = len(a)
dp(0,0)

428

In [24]:
path = []
state = (0,0)
while state is not None:
    path.append(state)
    state = prev[state]
print(path)

[(0, 0), (1, 0), (2, 2), (3, 2), (4, 2), (5, 1), (6, 0), (7, 0), (8, 0), (9, 2), (10, 2), (11, 1), (12, 0), (13, 2), (14, 1), (15, 0), (16, 2), (17, 2), (18, 1), (19, 0), (20, 0), (21, 0), (22, 0), (23, 2), (24, 1), (25, 0), (26, 2), (27, 1), (28, 0), (29, 2), (30, 1)]


Wir übersetzen die Abfolge der states in Aktionen:

In [26]:
actions = []
for i in range(len(path)-1):
    k1 = path[i][1]
    k2 = path[i+1][1]
    if (k1,k2) == (0,2): actions.append('K')
    elif (k1,k2) == (2,1): actions.append('V')
    elif (k1,k2) == (1,0): actions.append('C')
    else: actions.append('x')
if path[-1][1] == 2:
    actions.append('V')
else:
    actions.append('x')
print(actions)

['x', 'K', 'x', 'x', 'V', 'C', 'x', 'x', 'K', 'x', 'V', 'C', 'K', 'V', 'C', 'K', 'x', 'V', 'C', 'x', 'x', 'x', 'K', 'V', 'C', 'K', 'V', 'C', 'K', 'V', 'x']


Die Aktionen an jedem Tag

In [28]:
for x, y in zip(a,actions):
    print(x,y)

48 x
12 K
60 x
93 x
97 V
42 C
25 x
64 x
17 K
56 x
85 V
93 C
9 K
48 V
52 C
42 K
58 x
85 V
81 C
84 x
69 x
36 x
1 K
54 V
23 C
15 K
72 V
15 C
11 K
94 V
